<a href="https://colab.research.google.com/github/ashennavindu/Statistical-Learning-e23022/blob/main/Assignment_05/.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Foundations of Statistical Inference & Hypothesis Testing**

Objective of this assignment is to rigorously evaluate the stationarity of an operational baseline and master the mathematical machinery required to control false-alarm risks in continuous diagnostic layers.

## **Part A: Theoretical Fundamentals (Maximum Likelihood & Decision Space)**



1. Consider the sequence of i.i.d. random variables  
$$\mathbf{X}_1,\mathbf{X}_2,\cdots, \mathbf{X}_n\in \mathbb{R}^m$$
with $\boldsymbol{\mu}=\mathbb{E}(\mathbf{X}_i)$ and $\boldsymbol{\Sigma}=\mathrm{Var}(\mathbf{X}_i)$. The unobservable multivariate population parameters $(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ are estimated with their corresponding unbiased empirical sample estimators $(\widehat{\boldsymbol{\mu}}_n, \mathbf{S})$, where
$$\widehat{\boldsymbol{\mu}}_n = \frac{1}{n}\sum_{i=1}^n \mathbf{X}_{i},$$
and
$$\mathbf{S} = \frac{1}{n-1} \sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T .$$

* Formally derive why the raw Maximum Likelihood Estimator for the covariance matrix, $\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}$, is inherently **biased** in finite samples, by showing that:

$$\mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \frac{n-1}{n}\boldsymbol{\Sigma}$$


* Mathematically demonstrate how applying Bessel's correction yields the unique, unbiased estimator $\mathbf{S}$.


2. Under the null hypothesis $H_0$ that a structural asset remains completely healthy (the status quo), a chosen test statistic tracks a deterministic parametric distribution.
* Define a **Type I Error ($\alpha$)** and a **Type II Error ($\beta$)** explicitly within the context of continuous structural health monitoring.
* If an engineer sets an ultra-conservative significance threshold (e.g., $\alpha = 0.0001$) to prevent false alarms from shutting down an industrial plant, explain the mathematical consequence this decision exerts on the statistical power of the diagnostic layer. How does this modify the geometric volume of your "healthy operation" confidence ellipsoid?

1. Type I Error ($\alpha$) and Type II Error ($\beta$) in Structural Health Monitoring

Type I Error (False Alarm) - $\alpha$:
The system detects a fault even though the structure is healthy. This can cause unnecessary maintenance or shutdowns.

Type II Error (Missed Detection) - $\beta$:
The system fails to detect an actual fault and considers the structure healthy. This can lead to serious damage or failure.

2. Consequences of an Ultra-Conservative Significance Threshold ($\alpha = 0.0001$)

Effect on Statistical Power:
The statistical power ($1-\beta$) decreases, so the system becomes less likely to detect real faults.

Effect on Confidence Ellipsoid:
The confidence ellipsoid becomes larger, making it harder for observations to be classified as anomalies. This reduces false alarms but increases the chance of missing real problems.

## **Part B: Theoretical Extension (Asymptotic Distributions & Slutsky's Theorem)**

The Lindeberg-Lévy Multivariate Central Limit Theorem establishes that as the sample snapshot size approaches infinity ($n \to \infty$), the standardized difference between the empirical mean and population mean converges in distribution to a Multivariate Gaussian:

$$\sqrt{n}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \mathscr{N}(\mathbf{0}, \boldsymbol{\Sigma})$$



However, the true population covariance matrix $\boldsymbol{\Sigma}$ is a latent, unobservable parameter.
* State **Slutsky’s Theorem** rigorously.
* Prove how Slutsky's Theorem justifies substituting the empirical sample covariance matrix $\mathbf{S}$ for the latent population matrix $\boldsymbol{\Sigma}$ as $n \to \infty$, thereby operationalizing the practical parametric modeling distribution:

$$\widehat{\boldsymbol{\mu}}_n \sim \mathscr{N}\left({\boldsymbol{\mu}}, \frac{1}{n}\mathbf{S}\right)$$







### 1. Slutsky's Theorem

Slutsky's Theorem states that if a sequence of random variables $X_n$ converges in distribution to $X$, and another sequence $Y_n$ converges in probability to a constant $c$, then $Y_n$ can be replaced by $c$ in large samples.

---

### 2. Justification for Using the Sample Covariance Matrix

The sample covariance matrix $S$ is a consistent estimator of the population covariance matrix $\Sigma$. Therefore,

$$
S \xrightarrow{p} \Sigma
$$

According to the Multivariate Central Limit Theorem,

$$
\sqrt{n}(\hat{\mu}_n-\mu) \xrightarrow{d} N(0,\Sigma)
$$

Since $S$ converges in probability to $\Sigma$, Slutsky's Theorem allows us to replace the unknown covariance matrix $\Sigma$ with the sample covariance matrix $S$ when $n$ is large.

Hence,

$$
\hat{\mu}_n \sim N\left(\mu,\frac{1}{n}S\right)
$$

This makes the model practical because $S$ can be calculated from the data, while $\Sigma$ is unknown.

## **Part C: Numerical Verification**

***Scenario:** You are provided an industrial dataset containing $n = 5,000$ snapshots across $m = 3$ sensor channels measuring physical strain. To verify the weak i.i.d. assumption before initializing a global PCA or Factor Analysis layer, you must explicitly test **Condition 1: Constant Joint Mean (First Moment Homogeneity)** across the time index.*

**Your Task:**
Write a clean, self-contained Python function within Google Colab to partition the dataset into $g = 5$ consecutive, non-overlapping blocks (chunks) of size $n_j = 1,000$. Implement a Multivariate Analysis of Variance (MANOVA) tracking framework from scratch (or leveraging `statsmodels`) to calculate **Wilks' Lambda ($\Lambda$)**.

$$\Lambda = \frac{|\mathbf{W}|}{|\mathbf{B} + \mathbf{W}|}$$

Transform your calculated $\Lambda$ using **Bartlett’s $\chi^2$ approximation**:


$$\chi^2_{\text{calc}} = -\left(n - 1 - \frac{m + g}{2}\right) \ln \Lambda$$

Execute your code using the synthetic data generator provided below. Report your exact calculated Wilks' $\Lambda$, the Bartlett $\chi^2$ test statistic, and its corresponding $p$-value under $\chi^2 \sim \chi^2_{m(g-1)}$. Based on an alpha threshold of $\alpha = 0.05$, state your final diagnostic conclusion: Has the baseline center shifted over time, or is the first moment homogeneous?



In [8]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# --- Synthetic Strain Sensor Data Generator ---
np.random.seed(42)
n_samples = 5000
m_features = 3

# Generating base stationary multivariate normal noise
base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

# Injecting a subtle, hidden structural drift in the final 1,000 snapshots
base_data[4000:, 0] += 0.015  # Sensor 1 drift
base_data[4000:, 2] -= 0.010  # Sensor 3 drift

df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])

def verify_first_moment_homogeneity(df, g_chunks=5):
    n, m = df.shape
    n_j = n // g_chunks

    # 1. Partition into g chunks
    chunks = [df.iloc[i*n_j:(i+1)*n_j] for i in range(g_chunks)]

    # 2. Global mean and chunk means
    global_mean = df.mean().values
    chunk_means = [chunk.mean().values for chunk in chunks]

    # 3. Calculate Within-Chunk (W) and Between-Chunk (B) matrices
    W = np.zeros((m, m))
    B = np.zeros((m, m))

    for i in range(g_chunks):
        # Within-group sum of squares and cross-products
        diff_w = chunks[i].values - chunk_means[i]
        W += diff_w.T @ diff_w

        # Between-group sum of squares and cross-products
        diff_b = (chunk_means[i] - global_mean).reshape(-1, 1)
        B += n_j * (diff_b @ diff_b.T)

    # 4. Wilks' Lambda calculation
    # Lambda = |W| / |B + W|
    det_W = np.linalg.det(W)
    det_total = np.linalg.det(B + W)
    wilks_lambda = det_W / det_total

    # 5. Bartlett's Chi-Square Approximation
    chi2_calc = -(n - 1 - (m + g_chunks) / 2.0) * np.log(wilks_lambda)
    dof = m * (g_chunks - 1)
    p_value = 1 - stats.chi2.cdf(chi2_calc, dof)

    return {
        "Wilks_Lambda": wilks_lambda,
        "Chi2_Stat": chi2_calc,
        "p_value": p_value,
        "dof": dof,
        "conclusion": "Stationary (Homogeneous)" if p_value > 0.05 else "Shift Detected (Non-Homogeneous)"
    }

# Execute Analysis
results = verify_first_moment_homogeneity(df_strain)
for key, val in results.items():
    print(f"{key}: {val}")

Wilks_Lambda: 0.9904519796929591
Chi2_Stat: 47.9215049961488
p_value: 3.2255259508895406e-06
dof: 12
conclusion: Shift Detected (Non-Homogeneous)


# **Geometric Subspace Optimization via Principal Component Analysis (PCA)**

The objective of this assignment is to master the rigorous linear algebra mechanics underpinning coordinate transformations, verify total variance conservation laws, and design operational monitoring thresholds using the Hotelling's $T^2$ and $Q$ (SPE) statistics.

## **Part A: Theoretical Fundamentals (Coordinate Projections & Orthogonality)**

1. Let $\widetilde{\mathbf{X}}_i = (\mathbf{X}_i - \boldsymbol{\mu}) \in \mathbb{R}^m$ represent a zero-mean, centered baseline feature vector with a symmetric, positive-semidefinite population covariance matrix $\boldsymbol{\Sigma} = \mathbb{E}[\widetilde{\mathbf{X}}_i \widetilde{\mathbf{X}}_i^T]$. We apply the spectral decomposition $\boldsymbol{\Sigma} = \mathbf{P} \mathbf{\Lambda} \mathbf{P}^T$, and define the orthogonal projection mapping:

$$\mathbf{Z}_i = \mathbf{P}^T \widetilde{\mathbf{X}}_i$$


* Prove step-by-step that the theoretical covariance matrix of the transformed vector, $\mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T]$, simplifies exactly to the strictly diagonal matrix $\mathbf{\Lambda}$.
* Explain the precise geometric and statistical meaning of this diagonal result regarding the cross-covariances between distinct coordinates $Z_{i,j}$ and $Z_{i,k}$ (for $j \neq k$).


2. The total variance of a multivariate system can be measured in its raw coordinate space or its transformed principal component space.
* Utilizing the algebraic properties of the matrix trace ($\text{tr}$), specifically its invariance under cyclic permutations, formally prove that the total system variance is preserved during the mapping from raw space to the de-correlated principal space:

$$\text{tr}(\boldsymbol{\Sigma}) = \text{tr}(\mathbf{\Lambda}) = \sum_{j=1}^m \lambda_j$$


* If we partition the empirical matrix into an explained subspace ($\widehat{\mathbf{P}}_k \in \mathbb{R}^{m \times k}$) and an unexplained residual subspace ($\widehat{\mathbf{P}}_{m-k} \in \mathbb{R}^{m \times (m-k)}$), state the rigorous mathematical formulations for the **Cumulative Explained Variance Ratio $\Phi(k)$** and the **Residual Unexplained Variance Ratio $\Psi(k)$**.


3. In structural health monitoring (SHM), we truncate our coordinate system to retain only the first $k$ components ($k < m$). The centered raw vector is partitioned as $(\mathbf{x}_i - \widehat{\boldsymbol{\mu}}_n) = \widehat{\mathbf{P}}_k \mathbf{z}_{i,k} + \widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}$.
* Define the reconstructed vector $\widehat{\mathbf{x}}_i$ and the residual error vector $\mathbf{e}_i$.
* Prove via the Pythagorean identity that the total squared Euclidean norm of the centered observation vector cleanly decomposes into independent subspace energies:

$$\|\mathbf{x}_i - \widehat{\boldsymbol{\mu}}_n\|^2 = \|\mathbf{z}_{i,k}\|^2 + \|\mathbf{z}_{i,m-k}\|^2$$


* Contrast the diagnostic functions of Hotelling's $T^2$ statistic and the residual $Q$ statistic (SPE). If a structure encounters an environmental load anomaly (e.g., extreme operational wind loads) versus an internal structural fracture (e.g., a fatigue crack changing the structural load pathways), match each event type to the statistic ($T^2$ or $Q$) most sensitive to it and justify your choices.

### 1. Covariance of the Transformed Vector

Given

$$
Z_i=P^T\tilde{X}_i
$$

the covariance of $Z_i$ is

$$
E[Z_iZ_i^T]
=E[(P^T\tilde{X}_i)(P^T\tilde{X}_i)^T]
$$

$$
=E[P^T\tilde{X}_i\tilde{X}_i^TP]
$$

$$
=P^TE[\tilde{X}_i\tilde{X}_i^T]P
$$

$$
=P^T\Sigma P
$$

Using the spectral decomposition,

$$
\Sigma=P\Lambda P^T
$$

Therefore,

$$
P^T\Sigma P
=P^T(P\Lambda P^T)P
$$

$$
=(P^TP)\Lambda(P^TP)
=\Lambda
$$

Hence,

$$
E[Z_iZ_i^T]=\Lambda
$$

---

### 2. Meaning of the Diagonal Matrix

Since $\Lambda$ is diagonal,

$$
Cov(Z_{i,j},Z_{i,k})=0
\quad (j\neq k)
$$

This means different principal components are uncorrelated. Each component represents an independent direction of variation.

---

### 3. Preservation of Total Variance

The total variance is

$$
tr(\Sigma)
$$

Using $\Sigma=P\Lambda P^T$,

$$
tr(\Sigma)
=tr(P\Lambda P^T)
$$

Using cyclic invariance of trace,

$$
tr(P\Lambda P^T)
=tr(\Lambda P^TP)
$$

$$
=tr(\Lambda)
$$

Since $\Lambda$ is diagonal,

$$
tr(\Lambda)
=\sum_{j=1}^{m}\lambda_j
$$

Therefore,

$$
tr(\Sigma)=tr(\Lambda)=\sum_{j=1}^{m}\lambda_j
$$

The total variance is preserved.

---

### 4. Explained and Residual Variance Ratios

Cumulative Explained Variance Ratio:

$$
\Phi(k)
=
\frac{\sum_{j=1}^{k}\lambda_j}
{\sum_{j=1}^{m}\lambda_j}
$$

Residual Unexplained Variance Ratio:

$$
\Psi(k)
=
\frac{\sum_{j=k+1}^{m}\lambda_j}
{\sum_{j=1}^{m}\lambda_j}
$$

or

$$
\Psi(k)=1-\Phi(k)
$$

---

### 5. Reconstruction and Residual Error

The reconstructed vector is

$$
\hat{x}_i
=
\hat{\mu}_n+\hat{P}_kz_{i,k}
$$

The residual error vector is

$$
e_i
=
x_i-\hat{x}_i
$$

$$
=
\hat{P}_{m-k}z_{i,m-k}
$$

---

### 6. Pythagorean Energy Decomposition

Since the principal component subspaces are orthogonal,

$$
x_i-\hat{\mu}_n
=
\hat{P}_kz_{i,k}
+
\hat{P}_{m-k}z_{i,m-k}
$$

Therefore,

$$
\|x_i-\hat{\mu}_n\|^2
=
\|\hat{P}_kz_{i,k}\|^2
+
\|\hat{P}_{m-k}z_{i,m-k}\|^2
$$

Because orthogonal matrices preserve norms,

$$
\|x_i-\hat{\mu}_n\|^2
=
\|z_{i,k}\|^2
+
\|z_{i,m-k}\|^2
$$

This shows that the total energy is divided into explained and residual energies.

---

### 7. Hotelling's $T^2$ vs Residual $Q$ Statistic

**Hotelling's $T^2$ Statistic**

- Measures variation inside the principal component subspace.
- Sensitive to environmental or operational changes.

**Example:** Extreme wind loads.

---

**Residual $Q$ Statistic (SPE)**

- Measures variation outside the retained PCA subspace.
- Sensitive to new damage patterns not captured by the normal model.

**Example:** Fatigue crack or structural fracture.

---

### Conclusion

- **Environmental load anomaly → Hotelling's $T^2$**
- **Structural fracture/crack → Residual $Q$ (SPE)**

because environmental effects mainly change scores within the normal subspace, while structural damage creates new residual patterns outside the learned subspace.

## **Part B: Numerical Verification**



***Scenario:** You are building a real-time diagnostics layer for a structural component monitored by $m = 4$ sensors. You have a baseline archive of $n = 3,000$ nominal snapshots. To establish operational boundaries, you must write an optimization pipeline that analyzes how Hotelling’s $T^2$ and the residual $Q$ statistic respond to different truncation boundaries $k \in \{1, 2, 3\}$.*

**Your Task:**
Complete the Python class method below inside Google Colab. The function must:

1. Standardize and center the incoming realizations.
2. Decompose the unbiased sample covariance matrix $\mathbf{S}$ using `np.linalg.eigh` and sort components in descending order ($\widehat{\lambda}_1 \ge \dots \ge \widehat{\lambda}_m$).
3. Loop through every possible subspace cutoff value $k \in \{1, 2, 3\}$, calculating the sample vector array for Hotelling’s $T^2$ ($T^2_i = \sum_{j=1}^k \frac{z_{i,j}^2}{\widehat{\lambda}_j}$) and the $Q$ statistic ($Q_i = \sum_{j=k+1}^m z_{i,j}^2$).
4. Compute the expected value (empirical mean) of $T^2$ and $Q$ across all samples for each $k$.
5. Return these metrics to verify your theoretical derivations in Part A.



In [5]:
import numpy as np

class DiagnosticsLayer:

    def analyze_statistics(self, X):
        """
        X : array of shape (n_samples, m_features)

        Returns:
            results : dictionary containing the mean T² and Q
                      statistics for k = 1, 2, 3
        """

        # 1. Standardize and center data
        mu = np.mean(X, axis=0)
        sigma = np.std(X, axis=0, ddof=1)
        # Avoid division by zero if std is 0
        sigma[sigma == 0] = 1e-15
        X_std = (X - mu) / sigma

        # 2. Unbiased sample covariance matrix
        S = np.cov(X_std, rowvar=False, ddof=1)

        # Eigen-decomposition
        eigvals, eigvecs = np.linalg.eigh(S)

        # Sort in descending order
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        # PCA scores
        Z = X_std @ eigvecs

        results = {}

        # 3. Loop through k = 1, 2, 3
        for k in [1, 2, 3]:

            # Hotelling's T²
            T2 = np.sum((Z[:, :k] ** 2) / eigvals[:k], axis=1)

            # Residual Q statistic (SPE)
            Q = np.sum(Z[:, k:] ** 2, axis=1)

            # 4. Empirical means
            mean_T2 = np.mean(T2)
            mean_Q = np.mean(Q)

            results[k] = {
                "Mean_T2": mean_T2,
                "Mean_Q": mean_Q
            }

        return results

# **Latent Subspace Decomposition via Factor Analysis (FA)**

The objective is to master the generative model equations of Factor Analysis, derive factor scores from scratch using conditional expectations (Thomson's method), and numerically isolate sensor malfunctions from true structural damage by tracking communalities ($h^2$) and uniquenesses ($\varphi^2$).

## **Part A: Theoretical Fundamentals (Generative Model & Commonalities)**

1. Consider your structured data setup where $\mathbf{Z}_i \in \mathbb{R}^m$ is a normalized, zero-mean baseline feature vector whose population covariance matrix represents the universal correlation matrix $\mathbf{R} = \mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T]$. The Factor Analysis model posits the linear generative structure:
$$\mathbf{Z}_i = \boldsymbol{\Lambda} \mathbf{F}_i + \boldsymbol{\epsilon}_i$$
where $\mathbf{F}_i \in \mathbb{R}^k$ are the orthogonal common factors ($\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T] = \mathbf{I}_{k \times k}$) and $\boldsymbol{\epsilon}_i \in \mathbb{R}^m$ is the diagonalized measurement noise matrix ($\mathbb{E}[\boldsymbol{\epsilon}_i\boldsymbol{\epsilon}_i^T] = \boldsymbol{\Psi} = \text{diag}(\varphi_1^2, \dots, \varphi_m^2)$).
* Utilizing the constraint that the latent factors and errors are independent ($\mathbb{E}[\boldsymbol{\epsilon}_i \mathbf{F}_i^T] = \mathbf{0}$), prove the **Fundamental Equation of Factor Analysis**:

$$\mathbf{R} = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$$


* Focus on the $j$-th diagonal entry of this decomposition. Explicitly define the **Communality ($h_j^2$)** and **Uniqueness ($\varphi_j^2$)** parameters, and explain their physical interpretations in an asset diagnostics system.


2. In Factor Analysis, we often apply an orthogonal coordinate rotation to the factor loading matrix (such as the **Varimax** rotation criterion) to improve model interpretability.
* Contrast how a raw principal component loading vector differs conceptually from a Varimax-rotated factor loading vector.
* Explain why orthogonal rotations can alter the individual columns of the loading matrix $\boldsymbol{\Lambda}$ without changing the calculated communality ($h_j^2$) of any sensor or modifying the global covariance approximation $\mathbf{R} \approx \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$.



1. Because the common factors $\mathbf{F}_i$ are unobservable latent parameters, they cannot be directly extracted via geometric projections like PCA. Instead, we must treat them as hidden random vectors and calculate their realization using **Thomson’s Regression Method**, which relies on the conditional expectation $\mathbf{f}_i = \mathbb{E}[\mathbf{F}_i | \mathbf{z}_i]$.
* Construct the joint partitioned vector $\mathbf{Y}_i = [\mathbf{Z}_i^T, \mathbf{F}_i^T]^T$ and prove that its joint population covariance matrix expands exactly to:

$$\boldsymbol{\Sigma}_{YY} = \begin{bmatrix} \mathbf{R} & \boldsymbol{\Lambda} \\ \boldsymbol{\Lambda}^T & \mathbf{I}_{k \times k} \end{bmatrix}$$


* Under the standard assumption of a Multivariate Normal distribution, the conditional mean of a partitioned system is governed by the projection identity $\mathbb{E}[\mathbf{X}_2 | \mathbf{x}_1] = \boldsymbol{\mu}_2 + \boldsymbol{\Sigma}_{21} \boldsymbol{\Sigma}_{11}^{-1} (\mathbf{x}_1 - \boldsymbol{\mu}_1)$. Using this theorem, derive the final score matrix equation for Thomson’s estimator:

$$\mathbf{f}_i = \boldsymbol{\Lambda}^T \mathbf{R}^{-1} \mathbf{z}_i$$


### 1. Fundamental Equation of Factor Analysis

Given

$$
Z_i=\Lambda F_i+\epsilon_i
$$

the covariance matrix is

$$
R=E[Z_iZ_i^T]
$$

Substituting the factor model,

$$
R=E[(\Lambda F_i+\epsilon_i)(\Lambda F_i+\epsilon_i)^T]
$$

Expanding,

$$
R=E[\Lambda F_iF_i^T\Lambda^T]
+E[\Lambda F_i\epsilon_i^T]
+E[\epsilon_iF_i^T\Lambda^T]
+E[\epsilon_i\epsilon_i^T]
$$

Since

$$
E[F_iF_i^T]=I
$$

and

$$
E[\epsilon_iF_i^T]=0
$$

we obtain

$$
R=\Lambda I\Lambda^T+\Psi
$$

Therefore,

$$
R=\Lambda\Lambda^T+\Psi
$$

This is the Fundamental Equation of Factor Analysis.

---

### 2. Communality and Uniqueness

For the $j$-th variable,

$$
1=h_j^2+\phi_j^2
$$

where

$$
h_j^2=\sum_{l=1}^{k}\lambda_{jl}^2
$$

is the **communality**, and

$$
\phi_j^2
$$

is the **uniqueness**.

**Interpretation:**

- Communality: variance explained by common factors.
- Uniqueness: variance due to noise or sensor-specific effects.

In SHM, high communality means a sensor mainly reflects global structural behavior, while high uniqueness indicates local noise or independent effects.

---

### 3. PCA Loadings vs Varimax-Rotated Loadings

**PCA Loading Vector**

- Maximizes explained variance.
- Components may be difficult to interpret physically.

**Varimax-Rotated Factor Loading**

- Maximizes interpretability.
- Produces a simpler structure where variables load strongly on fewer factors.

---

### 4. Why Rotation Does Not Change Communality

Let $T$ be an orthogonal rotation matrix.

The rotated loading matrix is

$$
\Lambda^*=\Lambda T
$$

Then

$$
\Lambda^*(\Lambda^*)^T
=
(\Lambda T)(\Lambda T)^T
$$

$$
=
\Lambda TT^T\Lambda^T
$$

$$
=
\Lambda\Lambda^T
$$

since

$$
TT^T=I
$$

Therefore,

$$
R\approx\Lambda\Lambda^T+\Psi
$$

remains unchanged.

Also,

$$
h_j^2
=
\sum_{l=1}^{k}\lambda_{jl}^2
$$

is preserved because orthogonal rotations preserve vector length.

---

### 5. Joint Covariance Matrix

Define

$$
Y_i=
\begin{bmatrix}
Z_i \\
F_i
\end{bmatrix}
$$

Then

$$
\Sigma_{YY}
=
E[Y_iY_i^T]
$$

Expanding,

$$
\Sigma_{YY}
=
\begin{bmatrix}
E[Z_iZ_i^T] & E[Z_iF_i^T] \\
E[F_iZ_i^T] & E[F_iF_i^T]
\end{bmatrix}
$$

Using

$$
R=E[Z_iZ_i^T]
$$

and

$$
E[F_iF_i^T]=I
$$

Also,

$$
E[Z_iF_i^T]
=
E[(\Lambda F_i+\epsilon_i)F_i^T]
$$

$$
=
\Lambda E[F_iF_i^T]
+
E[\epsilon_iF_i^T]
$$

$$
=
\Lambda
$$

Therefore,

$$
\Sigma_{YY}
=
\begin{bmatrix}
R & \Lambda \\
\Lambda^T & I
\end{bmatrix}
$$

---

### 6. Thomson's Regression Estimator

For a multivariate normal system,

$$
E[X_2|x_1]
=
\mu_2+\Sigma_{21}\Sigma_{11}^{-1}(x_1-\mu_1)
$$

Since the variables are centered,

$$
\mu_1=\mu_2=0
$$

Using

$$
\Sigma_{21}=\Lambda^T
$$

and

$$
\Sigma_{11}=R
$$

gives

$$
f_i
=
E[F_i|z_i]
$$

$$
=
\Lambda^TR^{-1}z_i
$$

Therefore, Thomson's factor score estimator is

$$
f_i=\Lambda^TR^{-1}z_i
$$

## **Part B: Parametric Factor Partition Engine & UI Architecture**

The objective is to implement an end-to-end data processing and diagnostic visualization pipeline that instantiates a parametric **Factor Analysis (FA)** engine with **Varimax rotation**. The engine must partition empirical observation metrics into a shared structural subspace and a distinct diagonal noise floor, mapping the results onto a highly structured, asymmetric grid interface.

### Develop a Mathematical Engine & Pre-processing Guardrails


Implement an isolated algorithm to structure and decompose empirical data structures according to the following guidelines:

1. **Z-Score Mapping Layer:** Accept an arbitrary data frame and isolate numeric observation columns. Standardize the observations into standard normal space $Z \in \mathbb{R}^{n \times m}$ applying an unbiased sample standard deviation ($ddof=1$). Build an explicit numerical guardrail that catches zero-variance channels ($\sigma_j = 0$) and caps them to a minimum boundary threshold of $10^{-15}$ to prevent divide-by-zero runtime exceptions.
2. **Dimensionality Constraint Layer:** Let $m$ represent the count of isolated sensor features, and $k$ represent the target count of latent physical factors. Program two validation rule exceptions:
* Reject processing if the operational snapshot count ($n$) is less than or equal to the channel count ($m$).
* Throw an descriptive error if the requested latent space dimension ($k$) is not strictly less than the available feature space dimension ($m$).


3. **Information Decomposition Layer:** Fit a Factor Analysis framework utilizing a Maximum Likelihood or Singular Value Decomposition (SVD) algorithm combined with an orthogonal Varimax rotation matrix ($T$). From the fitted parameters, compute the vector arrays for:
* **Communality Vector ($h^2 \in \mathbb{R}^m$):** The total shared variance captured across each channel.
* **Uniqueness Vector ($\varphi^2 \in \mathbb{R}^m$):** The remaining isolated sensor noise floor.

In [7]:
import numpy as np
import pandas as pd
from sklearn.decomposition import FactorAnalysis


def varimax(Phi, gamma=1.0, q=20, tol=1e-6):
    """
    Orthogonal Varimax rotation.
    """
    p, k = Phi.shape
    R = np.eye(k)

    for _ in range(q):
        d_old = np.sum(np.diag(R))

        Lambda = Phi @ R

        u, s, vh = np.linalg.svd(
            Phi.T @ (
                Lambda**3
                - (gamma / p) * Lambda @ np.diag(np.sum(Lambda**2, axis=0))
            )
        )

        R = u @ vh
        d = np.sum(s)

        if d_old != 0 and d / d_old < 1 + tol:
            break

    return Phi @ R, R


class FactorAnalysisEngine:

    def fit(self, df, k):

        # --------------------------------------------------
        # 1. Z-Score Mapping Layer
        # --------------------------------------------------

        X = df.select_dtypes(include=[np.number]).to_numpy()

        n, m = X.shape

        mu = np.mean(X, axis=0)
        sigma = np.std(X, axis=0, ddof=1)

        # Zero-variance guardrail
        sigma = np.maximum(sigma, 1e-15)

        Z = (X - mu) / sigma

        # --------------------------------------------------
        # 2. Dimensionality Constraint Layer
        # --------------------------------------------------

        if n <= m:
            raise ValueError(
                f"Invalid dimensions: n={n} must be greater than m={m}."
            )

        if k >= m:
            raise ValueError(
                f"Invalid latent dimension: k={k} must be strictly less than m={m}."
            )

        # --------------------------------------------------
        # 3. Information Decomposition Layer
        # --------------------------------------------------

        fa = FactorAnalysis(
            n_components=k,
            random_state=42
        )

        fa.fit(Z)

        loadings = fa.components_.T

        # Varimax rotation
        loadings_rot, T = varimax(loadings)

        # Communality vector
        h2 = np.sum(loadings_rot**2, axis=1)

        # Uniqueness vector
        phi2 = 1.0 - h2
        phi2 = np.maximum(phi2, 0)

        return {
            "Z": Z,
            "Loadings": loadings_rot,
            "Rotation_Matrix": T,
            "Communality": h2,
            "Uniqueness": phi2
        }

### Latent Score Projection

1. **Mathematical State Reconstruction:** Using Thomson’s Minimum Mean Squared Error (MMSE) regression methodology, transform the standard normal snapshot array ($Z$) directly into hidden factor states $F \in \mathbb{R}^{n \times k}$ using the system relation:

$$F = Z R^{-1} \Lambda_{\text{rotated}}$$


2. **Performance Diagnostic Telemetry:** Calculate the empirical variance of the reconstructed factor scores enforcing an unbiased scale correction ($ddof=1$). Print a clean confirmation breakdown out to the logging console that presents the calculated system-wide metrics:

$$\text{Average System Communality \%} = \frac{1}{m}\sum_{j=1}^m h_j^2 \times 100$$


$$\text{Average System Uniqueness \%} = \frac{1}{m}\sum_{j=1}^m \varphi_j^2 \times 100$$

### Layout Engineering & Interface Design

Render a unified, production-grade **2 × 2 diagnostic canvas** using Plotly Subplots that enforces the following strict geometric layout constraints to ensure maximum readability and prevent overlapping label blocks:

* **Grid Specifications:** Configure the dashboard matrix with an explicit `horizontal_spacing=0.24` and a `vertical_spacing=0.28`. Set the global canvas parameters to `width=1250` and `height=750` utilizing a clean `plotly_white` theme.
* **Legend Formatting:** Place a single, horizontally oriented legend centered directly above the subplots (`orientation="h", x=0.5, y=1.02`). Enforce clean `margin` padding structures to isolate the interface elements (`t=150, b=60, l=140, r=80`).

Populate the grid layout using the four specific tracking profiles below:

```
                  2 × 2 FACTOR DIAGNOSTICS GRID MATRIX
+------------------------------------------+------------------------------------------+
|             [Row 1, Col 1]               |             [Row 1, Col 2]               |
|      Structural Loadings Heatmap         |     Horizontal Stacked Bar Chart         |
|   - Absolute values |λ_(j,r)|            |   - Shared Communality (h²) in Blue      |
|   - Color scale: 'YlOrRd'                |   - Isolated Uniqueness (φ²) in Orange   |
|   - Left-anchored scale bar (x = -0.15)  |   - Total space constraints: 0 to 100%   |
+------------------------------------------+------------------------------------------+
|             [Row 2, Col 1]               |             [Row 2, Col 2]               |
|         Sensor Noise Floor Line          |       Latent Empirical Variance          |
|   - Dot-dashed red line graph with markers|   - Vertical distribution bar chart      |
|   - Tracks exact Uniqueness values (φ²)  |   - Verifies energy level allocations    |
|   - Enforce tick angle adjustments (25°) |   - Black boundary outlines (width = 0.5)|
+------------------------------------------+------------------------------------------+

```

1. **Subplot (1,1): Structural Loadings Matrix Heatmap**
Render the absolute value of the rotated component weights matrix ($|\Lambda_{\text{rotated}}|$) using a continuous `YlOrRd` (Yellow-Orange-Red) color palette. To protect the clarity of the sensor text identifiers along the y-axis, isolate the continuous scale colorbar to the left side of the canvas by explicitly setting `x=-0.15`, `len=0.38`, `y=0.78`, `yanchor="middle"`, and `xanchor="right"`.
2. **Subplot (1,2): Variance Partitioning Profile**
Build a horizontal stacked bar chart mapping out the tracking allocation per sensor channel. The base layer must represent the percentage contribution of shared communalities ($h^2 \times 100$) utilizing color hex code `#1f77b4`. Stack the corresponding sensor uniqueness values ($\varphi^2 \times 100$) directly on top utilizing color hex code `#ff7f0e`. Force the x-axis limits strictly between `[0, 100]`.
3. **Subplot (2,1): Sensor Uniqueness Line Profile**
Construct a clear line chart tracking the isolated channel noise thresholds ($\varphi^2$) across all monitored hardware positions. Apply a red (`#d62728`), 2-pixel wide dot-dashed line string combined with visual `x` indicators. Apply a custom horizontal label layout optimization by rotating the category ticks to a distinct `tickangle=25` constraint.
4. **Subplot (2,2): Latent Factor Variance Distribution**
Draw a vertical bar chart displaying the empirical variance computed across the extracted common factor score vectors. Color the bars green (`#2ca02c`) with an explicit black border envelope (`width=0.5`).

# **Subspace Diagnostics & Feature De-correlation**

In structural health monitoring (SHM) and industrial asset diagnostics, multi-sensor grids generate highly continuous, collinear data streams. To establish reliable anomaly detection limits, engineers must project these high-dimensional observation spaces onto lower-dimensional sub-spaces.

This assignment evaluates two fundamental methods for dimensionality reduction using an identical 4-channel sensor array (`Sensor_1` to `Sensor_4`):

1. **Principal Component Analysis (PCA):** A geometric, non-parametric approach aimed at total variance maximization.
2. **Factor Analysis (FA):** A generative, parametric framework aimed at partitioning shared structural variance from unique localized sensor noise.



## **Part A: Develop a Feature Analysis Dashboard**

**UI Layout 1: PCA Optimization Suite ($2\times  3$ Horizontal Layout)**

The first interface must display a 5-panel horizontal grid layout.
* **Panel 1,1 (Heatmap):** Feature Loadings Matrix $|P_{j,r}|$.
* **Panel 1,2 (Bar Chart):** Absolute Eigenvalues ($\lambda_j$) tracking variance magnitude.
* **Panel 1,3 (Combined Bar + Line):** Marginal Explained Variance % per axis (bars) overlaid with Cumulative Captured Variance Curve (dashed line).
* **Panel 2,1 (Bar Chart):** Residual Space (Unexplained Variance %) showing excluded information ($100 - \text{Cumulative Variance \%}$).
* **Panel 2,2 (Scatter Line):** Sample Mean Hotelling's $T^2$ tracking distance vs. Subspace Size $k$.
* **Panel 2,3 (Scatter Line):** Sample Mean $Q$ Statistic (SPE) tracking residual decay vs. Truncation Cutoff $k$.

**UI Layout 2: FA Latent Subspace Suite ($2 \times 2$ Grid Layout)**

The second interface must structure its subplots into a balanced $2 \times 2$ matrix.

* **Panel 1,1 (Heatmap):** Structural Loadings Matrix $|\lambda_{j,r}|$.
* **Panel 1,2 (Stacked Horizontal Bar):** Variance Allocation breakdown showing Shared Communality ($h^2$) versus Channel Uniqueness ($\varphi^2$) totaling $100\%$ per sensor.
* **Panel 2,1 (Scatter Line):** Sensor Uniqueness Noise Floor Profile ($\varphi^2$) across monitored channels to instantly isolate instrument faults.
* **Panel 2,2 (Bar Chart):** Latent Factor Scores Empirical Variance showing the post-rotation power of the extracted physical dimensions.


## **Part B: Analysis**

### Generate the following synthetic data and answer the questions below

The synthetic multi-sensor asset data must be generated using
```python
import numpy as np
import pandas as pd

# Set execution seed for perfect reproducible properties
np.random.seed(42)
n_samples = 2500

# 1. Generate underlying, independent true physical driving forces (Latent Factors)
f1 = np.random.normal(0, 1, n_samples)  # Primary structural mode/load
f2 = np.random.normal(0, 1, n_samples)  # Secondary operational mode

# 2. Construct the Observable Multi-Sensor Array with targeted noise profiles
# Sensors 1 and 2 couple strongly to the primary process (f1)
s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)

# Sensor 3 couples strongly to the secondary process (f2)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)

# Sensor 4 suffers an extreme electrical/instrumentation fault (massive localized white noise)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)

# 3. Compile into the final evaluation DataFrame
df_asset = pd.DataFrame(
    data=np.vstack([s1, s2, s3, s4]).T,
    columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4']
)

```

**Empirical Covariance and Correlation Properties**

* **Sensors 1, 2, and 3:** High cross-correlation due to strong coupling with the common underlying physical structures ($f_1$ and $f_2$).
* **Sensor 4:** Highly un-correlated with all other channels, but exhibits an exceptionally large variance magnitude ($\sigma^2 \approx 2.0$) due to its high noise contamination scale.


### Question 1: The Total Variance Illusion (PCA Subplots 1 & 2 vs. FA Subplots 1 & 2)

* **Context:** In your PCA dashboard, `PC 1` captures roughly $45\%$ of the total system variance, and `PC 2` captures over $30\%$, indicating a highly structured, multi-component system. However, looking at the Factor Analysis dashboard, the **Variance Partitioning Stacked Bar** reveals that `Sensor_4` has a **Uniqueness value ($\varphi^2$) exceeding $98\%$**.
* **Core Problem:** 1. What does a Uniqueness value close to $100\%$ tell you about the physical nature of the data coming from `Sensor_4`?
2. Because PCA maximizes total global variance without distinguishing between types of variance, how does the massive localized noise of `Sensor_4` "trick" the PCA engine into inflating its top eigenvalues?
3. Why is this a major risk if an engineer relies *only* on PCA to build an automated plant anomaly detection framework?



### Question 1: The Total Variance Illusion

**1. What does a Uniqueness value close to 100% mean?**

A Uniqueness value ($\phi^2$) close to 100% means that almost all the variance of Sensor_4 is unique to that sensor and is not explained by the common factors. This suggests the sensor is mainly measuring noise, local disturbances, or sensor-specific effects rather than actual system behavior.

---

**2. How does Sensor_4 affect PCA?**

PCA maximizes total variance without separating useful variance from noise. Since Sensor_4 contains very large variance, PCA treats this variance as important information. As a result, the large noise contribution increases the leading eigenvalues and can influence the principal components.

---

**3. Why is this risky for anomaly detection?**

If an engineer relies only on PCA, the model may learn noise patterns instead of true structural behavior. This can lead to false alarms, missed detections, and poor diagnostic performance because the dominant principal components may be driven by sensor noise rather than real plant conditions.

### Question 2: Decoupling Structural Loading via Rotation (FA Subplot 1 vs. PCA Eigenvectors)

* **Context:** Look at the **Structural Loadings Matrix** heatmap in the top-left of the FA dashboard. You can see that `Sensor_1` and `Sensor_2` load heavily on `Factor 1`, while `Sensor_3` loads exclusively on `Factor 2`.
* **Core Problem:**
1. Traditional PCA forces its principal axes to follow a strict mathematical hierarchy ($\lambda_1 > \lambda_2 > \dots$) where every component tries to sweep up as much mixed variance as possible across all sensors. Explain how the **Varimax rotation** in Factor Analysis relaxes this rule to create "simple structure" (clean, isolated groupings).
2. From a plant operator's standpoint, why is the rotated FA heatmap much easier to troubleshoot during a structural failure than a raw, unrotated PCA eigenvector matrix?





### Question 2: Decoupling Structural Loading via Rotation

**1. How does Varimax rotation create a simple structure?**

PCA components are chosen to maximize variance, so each component may contain mixed contributions from many sensors. Varimax rotation redistributes the factor loadings to make them more distinct. As a result, each sensor tends to load strongly on one factor and weakly on the others, creating clear and isolated groupings.

---

**2. Why is the rotated FA heatmap easier to troubleshoot?**

The rotated FA heatmap clearly shows which sensors belong to each factor. For example, Sensor_1 and Sensor_2 may represent one structural loading pattern, while Sensor_3 represents another. During a structural failure, an engineer can quickly identify the affected factor and associated sensors.

In contrast, PCA eigenvectors often contain mixed loadings across many sensors, making it difficult to determine the physical source of the problem. Therefore, Factor Analysis provides better interpretability for diagnostics and troubleshooting.

### Question 3: Determining Subspace Truncation ($k$) using $T^2$ and $Q$ Profiles

* **Context:** Look at the right side of the PCA dashboard, showing **Mean Hotelling's $T^2$** and **Mean $Q$ Statistic (SPE)** plotted against the truncation cutoff $k$.
* **Core Problem:**
1. Describe the exact behavior of the Mean $Q$ Statistic curve as the subspace cutoff transitions from $k=1$ to $k=2$, and then from $k=2$ to $k=3$.
2. Why does a sharp drop followed by a distinct "flat elbow" in the $Q$ statistic identify **$k=2$** as the true hidden physical dimensionality of this asset?
3. If you incorrectly choose a truncation cutoff of $k=3$ for your monitoring system, what type of data are you forcing into your "clean" principal subspace?

### Question: Determining Subspace Truncation \(k\) using T² and Q Profiles

#### 1. How does the Mean Q Statistic behave as \(k\) increases?

- From **\(k = 1\) to \(k = 2\)**, the Mean Q Statistic drops sharply, showing that the second component captures important process information.
- From **\(k = 2\) to \(k = 3\)**, the curve becomes nearly flat, indicating little additional information is captured.

---

#### 2. Why does the sharp drop and flat elbow identify \(k = 2\) as the true dimensionality?

- The large drop from **\(k = 1\) to \(k = 2\)** shows that the second component contains meaningful physical information.
- The flat elbow after **\(k = 2\)** indicates that further components mainly represent noise.
- Therefore, **\(k = 2\)** is the true hidden physical dimensionality of the asset.

---

#### 3. What happens if \(k = 3\) is chosen incorrectly?

- Noise and residual variations are included in the "clean" principal subspace.
- The model may treat noise as meaningful information.
- This can reduce fault detection and monitoring performance.

### Question 4: Operational Trade-offs in System Health Monitoring

* **Context:** Imagine you are deploying a real-time predictive maintenance pipeline for a fleet of identical industrial assets based on these exact models.
* **Core Problem:** Compare the operational trade-offs of these two approaches:
1. **The PCA Strategy:** Tracking the main principal subspace using Hotelling's $T^2$ alongside a residual noise floor monitor using the $Q$ statistic.
2. **The FA Strategy:** Monitoring the rotated Latent Factor Scores directly.


* Which strategy is more robust against a single sensor losing calibration or experiencing an electrical short? Justify your choice using the **Sensor Uniqueness Noise Floor ($\varphi^2$)** metrics.

### Question: Operational Trade-offs in System Health Monitoring

#### 1. PCA Strategy vs FA Strategy

- **PCA Strategy** monitors the principal subspace using **Hotelling's T²** and detects unexplained variations using the **Q Statistic (SPE)**.
- **FA Strategy** monitors the rotated **Latent Factor Scores**, where each factor is linked to a specific physical loading pattern.

---

#### 2. Which strategy is more robust to a faulty sensor?

The **FA Strategy** is generally more robust against a single sensor losing calibration or experiencing an electrical short.

This is because Factor Analysis explicitly models sensor-specific noise through the **Sensor Uniqueness Noise Floor (\(\phi^2\))**. A faulty sensor's abnormal variation is largely absorbed into its uniqueness term rather than affecting all latent factors.

---

#### 3. Why do the \(\phi^2\) metrics help?

- A high **\(\phi^2\)** value indicates that a sensor contains more unique noise and less shared information.
- Sensor faults are therefore isolated more effectively.
- In PCA, a faulty sensor can distort multiple principal components because PCA does not explicitly separate unique sensor noise from shared system behavior.

Therefore, **FA provides better fault isolation and robustness**, while **PCA is better for overall anomaly detection and variance monitoring.**